## Importing Dependencies

In [ ]:
## For some arithmetic and Matrix Operations
import numpy as np

## Dataframe Manipulation
import pandas as pd

## For Visualization
import matplotlib.pyplot as plt

## For Visualization too
import seaborn as sns

## Creating Pipeline
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline


## Creating a function transformer
from sklearn.preprocessing import FunctionTransformer

## For Column Transformer
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector


## For preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

## For missing values
from sklearn.impute import SimpleImputer

## Getting the recall score on our train set
from sklearn.metrics import recall_score

## Getting the accuracy score on train set
from sklearn.metrics import accuracy_score

## Getting the classification report from our train set
from sklearn.metrics import classification_report

## Cross validation
from sklearn.model_selection import cross_val_score

## Gridsearch CV
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

## Imbalanced pipeline and SMOTE
from imblearn.pipeline import Pipeline, make_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks

## Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier


## Creating Functions

In [ ]:
## printing the shape and head
def head(df,shape_only=False):
    print(df.shape)

    if shape_only:
        return
    else:
        return df.head()

## for EDA of categorical values
def eda_bivariate_categorical(df,column,target):

    fig,ax = plt.subplots(figsize = (9,8))

    color = 'Set2'

    palette_color = sns.color_palette(color)

    ax = sns.countplot(x = column, data=df, hue=target,palette=color,order = df[column].value_counts().index)
    ax.set_ylabel('Count')

    offset = df[column].value_counts().max() * 0.005

    list_bars = df.groupby([column,target])[column].agg(['count']).unstack().fillna(0).values

    patches = ax.patches
    bars_pos = 0

    for i in range(df[target].nunique()):
        for j in range(df[column].nunique()):
            list_bars_col = list_bars[j] 
            total_sum = list_bars_col.sum()
            value = list_bars_col[i]

            percentage = value / total_sum

            if percentage == 0:
                bars_pos += 1
                continue
            else:
                x = patches[bars_pos].get_x() + patches[j].get_width()/2
                y = patches[bars_pos].get_height() + offset
                ax.annotate('{:.1f}%'.format(percentage*100), (x, y), ha='center')
                bars_pos += 1
    plt.show()

## Function that plots numerical variables into histogram and violin plot
def eda_bivariate_numerical(data,column,target,color,
                    figsize=(12,6),
                    # save=True,
                    val=0,
                    target_type = 'Numerical'):

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    cmap = sns.color_palette(color)
    val = val

    for i in range(1):
        for j in range(2):
            if j==0:
                    sns.histplot(data = data,x=data[column],hue=target,
                                bins=50,kde=True,palette=color,ax=axes[j])
                    axes[j].set(xlabel=None)
                    axes[j].grid(False)
            elif j==1:
                sns.boxplot(data = data,x=data[column],y = target, ax=axes[j], palette=color,orient='h',
                )
                axes[j].set(xlabel=None)
                axes[j].grid(False)
                val += 1
                plt.tight_layout()
            if target_type == 'Numerical':
                plt.suptitle(column)
            else:
                plt.suptitle(f'{column} vs. {target}')
    plt.show()
    
    # path = 'Figures\\Numerical\\'
    # if save:
    #     plt.savefig(f"{path}{column}.pdf",dpi=1000)


## print text to see the font
def print_text(text):
    fig, ax = plt.subplots(figsize=(6, 1), facecolor="#eefade")
    ax.text(0.5, 0.5, text, ha='center', va='center', size=40)
    ax.axis("off")
    plt.show()


## Reading Dataset and Showing its Description

In [ ]:
df = pd.read_csv('/kaggle/input/cardiovascular-diseases-risk-prediction-dataset/CVD_cleaned.csv')

## Viewing the dataframe and shape
head(df,shape_only=False) 

In [ ]:
## Setting the target variable
target = 'Heart_Disease'

In [ ]:
## Creating numerical and categorical columns
numerical = df.select_dtypes(include=['float64']).columns.sort_values()
categorical = df.select_dtypes(include=['object']).columns.sort_values()

## Printing the length of numerical and categorical. The total length should have
## the same length as our dataframe
print(f'There are {len(categorical)} Categorical variables')
print(f'There are {len(numerical)} Numerical variables')

In [ ]:
## Showing the columns in alphabetical order
df.columns.sort_values()

## Showing the descriptions of numerical variables
print('')
num_describe = df.describe().T
num_describe_table = num_describe.loc[:,['mean', 'std', '25%', '50%', '75%']]
print(num_describe_table)

## Showing the descriptions of categorical variables
print('')
object_describe_table = df.describe(include=object)
print(object_describe_table)
## print it to latex
# print(object_describe_table.to_latex())

If you want to change the fonts in matplotlib. Optional

In [ ]:
params = {'font.size' : 14,
          'font.family' : 'Libre Caslon Text',
          }
plt.rcParams.update(params)

## you can just use this to reset the font to default
# plt.rcdefaults()

## You can view how the font looks like thru here:
print_text("Hello World! 01")


## Exploratory Data Analysis (EDA)

### Target Variable

#### Categorical

In [ ]:
fig,ax = plt.subplots(figsize = (9,8))
color = 'Set2'
palette_color = sns.color_palette(color)

ax = sns.countplot(x = target, 
                data=df,
                palette=color,
                order = df[target].value_counts().index
                )
ax.set_ylabel('Count')

patches = ax.patches

for j in range(len(patches)):
        percentage = list(df[target].value_counts())[j]/df[target].value_counts().sum()
        offset = df[target].value_counts().max() * 0.01
        x = patches[j].get_x() + patches[j].get_width()/2
        y = patches[j].get_height() + offset
        ax.annotate('{:.1f}%'.format(percentage*100), (x, y), ha='center')

plt.show()
    

- We can deduct from this that the target variable is imbalanced

### Univariate Analysis

In [ ]:
for i in df.columns:
    
    if i == target:
        continue

    if i in categorical:
        if df[i].nunique() > 15:
            print(f'column {i} has many unique values n = {df[i].nunique()} and will not be plotted')
            print('=======================================================')
            continue
        else:
            print(f'{i}')
            fig,ax = plt.subplots(figsize = (9,8))
            color = 'Set2'
            palette_color = sns.color_palette(color)
            ax = sns.countplot(x = i, 
                data=df,
                palette=color,
                order = df[i].value_counts().index
                )
            ax.set_ylabel('Count')

            patches = ax.patches

            for j in range(len(patches)):
                # list_unq_val = list(df[i].unique())

                # cleaned = [x for x in list_unq_val if str(x) != 'nan']
                offset = df[i].value_counts().max() * 0.01
                percentage = list(df[i].value_counts())[j]/df[i].value_counts().sum()
                x = patches[j].get_x() + patches[j].get_width()/2
                y = patches[j].get_height()+ offset
                ax.annotate('{:.1f}%'.format(percentage*100), (x, y), ha='center')
                
            plt.show()
            print('=======================================================')
    
    elif i in numerical:
        print(f'{i}')
        fig,ax = plt.subplots(figsize = (9,8))
        color = 'Set2'
        palette_color = sns.color_palette(color)
        ax = sns.histplot(x = i, 
                data=df,
                # bins = 'auto',
                # bins = 50,
                kde = True,
                color=palette_color[0],
                )
        ax.set_ylabel('Count')
        plt.show()
        print('=======================================================')




### Bivariate Analysis for Classification

#### Categorical

In [ ]:
for i in categorical:
        if i == target:
            continue

        if df[i].nunique() > 15:
            print(f'column {i} has many unique values n = {df[i].nunique()} and will not be plotted')
            print('=======================================================')
            continue

        if i in df.columns:
            print(f'{i} vs. {target}')
            eda_bivariate_categorical(df,i,target)
            print('=======================================================')

#### Numerical

In [ ]:
for i in numerical:
        if i == target:
            continue

        if i in df.columns:
            print(f'{i} vs. {target}')
            eda_bivariate_numerical(
                    data = df,
                    column = i,
                    target = target,
                    color = 'Set2',
                    figsize=(12,7.5),
                    val=0)
            print('=======================================================')

### Multivariate Analysis

In [ ]:
## Plotting the correlation matrix
correlation_matrix = df[numerical].corr()
plt.figure(figsize=(9,8))

## use mask to cover the upper diagonal in the matrix
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix,
            cmap='RdBu_r',
            # cmap='RdYlGn',
            annot=True,
            # Masking the diagonal
            # mask=mask,
            fmt='.2f',
            vmin=-1, vmax=1)

## Saving the figure
# plt.savefig("latex2.pdf")

plt.show()

## Preprocessing 

Changing the values of Heart Disease to 0 and 1 for preprocessing steps

In [ ]:

df['Heart_Disease'] = df['Heart_Disease'].map({'No':0,'Yes':1})
print('')
print(df['Heart_Disease'].value_counts())

Splitting the train and test set. Using stratify to keep the ratio between two classes be the same

In [ ]:
from sklearn.model_selection import train_test_split

train,test = train_test_split(df, test_size=0.2,random_state=22,stratify=df['Heart_Disease'])

print(train.shape)
print(test.shape)

Showing the ratio of the target variable from train and test set

In [ ]:
yes = train['Heart_Disease'].value_counts()[0]/len(train['Heart_Disease'])*100
no = train['Heart_Disease'].value_counts()[1]/len(train['Heart_Disease'])*100
print('Train Set')
print(f'ratio of people with heart disease to total is {yes}')
print(f'ratio of people that dont have heart disease to total is {no}')
print('')

yes = test['Heart_Disease'].value_counts()[0]/len(test['Heart_Disease'])*100
no = test['Heart_Disease'].value_counts()[1]/len(test['Heart_Disease'])*100
print('Test Set')
print(f'ratio of people with heart disease to total is {yes}')
print(f'ratio of people that dont have heart disease to total is {no}')

In [ ]:
## Splitting the X and y variables in the train set
X_train = train.drop("Heart_Disease", axis=1)
y_train = train["Heart_Disease"].copy()

## Splitting the X and y variables in the test set
X_test = test.drop("Heart_Disease", axis=1)
y_test = test["Heart_Disease"].copy()

Printing the number of unique values per each column

In [ ]:
X_train.nunique()

Notes:

- There are 8 categorical variables. Variables that are not in order
- There are 7 numerical variables.
- There are 3 ordinal variables. The General Health, Age Category, and the Checkup variable. The data from this can be represented with an order.


### Creating Pipelines

#### Categorical Pipeline

In [ ]:
cat_pipeline = make_pipeline(OneHotEncoder(handle_unknown='ignore',drop='first'))

- For categorical pipeline, only OneHotEncoder will be implemented. Since this data set has been cleaned and there are no missing values

#### Numerical Pipeline

In [ ]:
num_pipeline = make_pipeline(
                             FunctionTransformer(np.log1p,feature_names_out='one-to-one'),
                             StandardScaler()
                            )   

For numerical pipeline, two methods are used:

1. Log Transform: From the EDA, most of the numerical functions are skewed right. Taking the log(x+1) of the variable will help fix the distribution
2. Standard Scaler: The numerical variables will be scaled to put them all on the same scale

#### Ordinal Pipelines

In [ ]:
## Age Category Pipeline
agecat_pipeline = make_pipeline(
        OrdinalEncoder()
)

## General Health Pipeline
genhealth_pipeline = make_pipeline(
        OrdinalEncoder(categories=[['Poor','Fair','Good','Very Good','Excellent']])
)

## Checkup Pipeline
checkup_pipeline = make_pipeline(
        OrdinalEncoder(categories=[['Within the past year','Within the past 2 years','Within the past 5 years','5 or more years ago','Never']])
)

- For ordinal variables, the variables are transformed based on their order. The values with in the lowest order will start with 0 and increases by 1.

#### Creating the pipeline lists

In [ ]:
## Setting each column to the pipeline where they will be used
num_pipe_col = numerical

cat_pipe_col = ['Arthritis', 'Depression', 'Diabetes',
       'Exercise', 'Other_Cancer', 'Sex',
       'Skin_Cancer', 'Smoking_History']

#### Finalizing the preprocessing pipeline

In [ ]:
## Combining all the pipelines and creating a main pipeline to enter all the data
preprocessing = ColumnTransformer([
    ('Categorical', cat_pipeline,   cat_pipe_col),
    ('Age_Category',agecat_pipeline,['Age_Category']),
    ('Checkup',checkup_pipeline,['Checkup']),
    ('Gen_health',genhealth_pipeline,['General_Health']),
    ('Numerical',   num_pipeline,  num_pipe_col),
],remainder='passthrough')
preprocessing

In [ ]:
## Using preprocessing pipeline
print('Shape before the preprocessing:')
print(X_train.shape)

train_preprocessed = preprocessing.fit_transform(X_train)

print('Shape after the preprocessing:')
print(train_preprocessed.shape)

## Model Training

Creating a stratified Kfold for cross validating the test set in different machine learning models

In [ ]:
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=10,shuffle=True,random_state=22)

Specifying the model names

In [ ]:
models = {
    'Logistic_Regression':LogisticRegression(max_iter=10000,random_state=22),
    'Decision Tree':DecisionTreeClassifier(random_state=22),
    'Random_Forest':RandomForestClassifier(n_estimators=100,random_state=22),
    'K-Nearest_Neighbor':KNeighborsClassifier(),
    'GaussianNB':GaussianNB(),
}

scores_dict = {}

report_dict = {}

Trying all the models and getting their cross validation score and showing its classification report.

In [ ]:
for model_name,model in models.items():
    model_pipeline = make_pipeline(preprocessing,
                              SMOTE(random_state=22),
                              model  
                                )
    scores = cross_val_score(model_pipeline, 
                            X_train, 
                            y_train, 
                            scoring='f1', 
                            cv=kf,
                            # verbose=1,
                            n_jobs=-1,
                            )
    model_score_mean = np.mean(scores)
    scores_dict[model_name] = model_score_mean
    print('------------------------------------------------------------')
    print(f'The score for {model_name} is {model_score_mean}')

    ## fitting the pipeline for classification report
    model_pipeline.fit(X_train,y_train)

    prediction = model_pipeline.predict(X_train)

    report = classification_report(y_train, prediction, output_dict=True)
    report_dict[model_name] = report
    print('')
    print(f'This is the classification report for {model_name}:')
    report_df = pd.DataFrame(report).T
    print(report_df)
    print('------------------------------------------------------------')
    

- **Note:** The KF fold are used for the cross validation process. The model training is combined already with the cross validation to prevent the bias or the overfitting of the training set.

- **Additional Note:** Included in the pipeline for model training is the SMOTE. SMOTE is used to deal with the imbalances classes. SMOTE is included in the pipeline where, the data only after being cross validated will then be subjected to SMOTE to prevent the data leakage among training and testing sets.

## Thanks for getting up to here! if you have any questions or clarification you can comment it down below. Cheers!